# 02 前処理の点検 —— 党を見分ける手がかりが残っていないか

**問い**: 前処理した会議録の本文に、立場とは関係なく「どの党の発言か」が分かってしまう手がかり(党名・会派名・自己紹介・人の名前・手続きの決まり文句など)が残っていないか。

**なぜ大事か**: 本文から党の立場を測るとき、党名などが残っていると、「立場の違い」ではなく「党名の違い」を拾ってしまう。

**方法**: 党ごとの特徴語(その党が、ほかの党より目立って多く使う言葉)を出し、上位に手がかりが出てこないかを見る。手がかりになりそうな言葉は、件数を数え、文脈ごと読んで確かめる。

**このノートの成り立ち**: もともとは、特徴語を出す → 手がかりを見つける → 前処理の決まりを直す、をくり返して前処理の決まりを作った作業用のノートだった。決まりは `src/preprocess/`・`scripts/02_preprocess.py` に移したので、今は**直した前処理で上から順に実行し、手がかりが消えたかを点検するノート**にした。各節の「見つけたこと」に、最初の作業で何が見つかり、どう直したかを書いた。

**結論(先に)**
- 党名・会派名・自己紹介は「政党」に置き換わり(マスク)、委員長報告・司会役の決まり文句は 0件になった
- 一般の言葉と重なる略称(中道・国民・参政・みらい)は、党を指す文脈のときだけ置き換えている。「中道的な」のような一般の言葉は残る(正しい)
- 隠した後の特徴語は、**政策の話題**が中心になった(国民「ガソリン」「暫定税率」、維新「副首都」「無償化」、参政「外国人」など)
- ただし、**話し方の違い**(自民「であります」、共産「ですよ」)は残る → 05 で、BERT の埋め込みの主な違いが話し方だったことにつながる

**流れ**: 準備 → 1. 読み込みと会派 → 党 → 2. 特徴語で手がかりを探す → 3. 委員長報告 → 4. 一般の言葉と重なる略称 → 5. 人の名前 → 6. 気になった言葉を文脈で確かめる → 7. 会派 → 党の対応表

※ 2026-10-25 に上から順に実行し直した。

## 準備

`src/` の部品を読み込む。
- `filter_speeches`: 発言でないもの・議員以外の発言・委員長報告などを除く / `prepare_text`: 本文から見出しを外し、党名・人の名前などを隠す / `mask_parties`: 党名・会派名を「政党」に置き換える(すべて `src/preprocess/filters.py`)
- `fightin_words`・`top_terms`: 党ごとの特徴語を出す(`src/analysis/keywords.py`)
- `build_member_table`: 議員の氏名と名字の一覧を作る(`src/preprocess/members.py`)

In [1]:
import sys, json, re
from pathlib import Path
import pandas as pd
import numpy as np
sys.path.insert(0, "..")
from src.preprocess.filters import filter_speeches, prepare_text, mask_parties
from sklearn.feature_extraction.text import CountVectorizer
from src.analysis.keywords import fightin_words, top_terms
from src.preprocess.members import build_member_table
import src.preprocess.filters as f
from src.preprocess.filters import SURNAME_RE

## 1. 読み込みと、会派 → 党

取得した会議録をすべて読み込み、除く発言を除いて、会派の名前から党を決める(対応表 `group_party_map.csv` を使う)。本文は、隠したもの(`text`)と隠していないもの(`text_nomask`)の2つを作る。

In [2]:
# 会派 → 党の対応表(前処理 scripts/02_preprocess.py と同じものを読む)
GROUP2PARTY = pd.read_csv("../data/external/group_party_map.csv").set_index("speakerGroup")["party"].to_dict()

# 読み込み
RAW = Path("../data/raw/kokkai")
dfs = []
for p in sorted(RAW.glob("speech_*.json")):
    with open(p, encoding="utf-8") as fh:          # f は filters の部品の名前なので使わない
        dfs.append(pd.DataFrame(json.load(fh)["speechRecord"]))
df = pd.concat(dfs, ignore_index=True)
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.to_period("M")

# 除外
q = filter_speeches(df)
q = q[q["date"] < "2026-08-01"].copy()

# 政党(本文を作る前に)
q["party"] = q["speakerGroup"].map(GROUP2PARTY).fillna("その他")

# 本文(マスクあり / なし)
q["text"] = [prepare_text(s, sp, p) for s, sp, p in zip(q["speech"], q["speaker"], q["party"])]
q["text_nomask"] = q["speech"].map(lambda s: prepare_text(s, mask=False))
print("全体:", len(df), "→ 除外後:", len(q))

全体: 194567 → 除外後: 88297


**結果**: 全体 194,567件 → 除いた後 **88,297件**。

(最終版の前処理 `scripts/02_preprocess.py` では、さらに、党を手で指定する表・除く表・手続きの発言を除く決まり(05 で見つけたもの)を使うので、`speeches.parquet` は 88,015件になる。)

次に、院ごとの会派の名前と件数を見る。

In [3]:
q.groupby(["nameOfHouse", "speakerGroup"]).size().sort_values(ascending=False)

nameOfHouse  speakerGroup
衆議院          立憲民主党・無所属       12606
参議院          立憲民主・社民・無所属      6802
衆議院          国民民主党・無所属クラブ     6558
参議院          国民民主党・新緑風会       5655
衆議院          自由民主党・無所属の会      5325
             中道改革連合・無所属       5274
             日本維新の会           5074
参議院          公明党              4886
             立憲民主・無所属         4718
             日本維新の会           4676
             日本共産党            4303
衆議院          日本共産党            3154
参議院          自由民主党            2322
             参政党              2161
衆議院          公明党              1946
参議院          自由民主党・無所属の会      1924
衆議院          参政党              1916
             れいわ新選組           1843
参議院          れいわ新選組           1794
衆議院          チームみらい           1244
参議院          各派に属しない議員         948
衆議院          有志の会              745
参議院          沖縄の風              531
             日本保守党             389
             社会民主党             388
衆議院          無所属               248
参議院          チームみらい・無所属の会      201
衆議院          日本保守党           

**結果**: 同じ党でも、院によって会派の名前が違う(立憲: 衆議院「立憲民主党・無所属」、参議院「立憲民主・社民・無所属」「立憲民主・無所属」)。→ 1つの党に、いくつもの会派の名前を対応させる必要がある(対応表は 7. で表示)。

次のセルは、党名・会派名を隠す関数の試し。

In [4]:
print(mask_parties("国民民主党・新緑風会の河合です。N国党と自公政権。"))

政党の河合です。政党と政党政権。


**結果**: 「国民民主党・新緑風会の河合です。N国党と自公政権。」→「政党の河合です。政党と政党政権。」。略称(N国党・自公)も隠せている。

党ごとの件数を数える。

In [5]:
q["party"].value_counts()

party
立憲     24267
国民     12288
維新      9760
自民      9588
共産      7457
公明      6841
中道      5296
参政      4088
れいわ     3637
その他     2647
みらい     1451
保守       589
社民       388
Name: count, dtype: int64

**結果**: 立憲 24,267・国民 12,288・維新 9,760・自民 9,588・共産 7,457・公明 6,841・中道 5,296・参政 4,088・れいわ 3,637・みらい 1,451・保守 589・社民 388。対応表に無い会派(各派に属しない議員・有志の会・沖縄の風など)は「その他」(2,647)。

## 2. 党ごとの特徴語で、手がかりを探す

**Fightin' Words**(Monroe ほか 2008)で、党ごとに「ほかの党より目立って多く使う言葉」を出す。言葉は2〜4文字の並び(日本語の単語分けをせず、文字の並びで数える)。分析する党の発言から 2万件を無作為に選ぶ。

見るもの: 各党の上位15個に、党名・人の名前・決まり文句など、立場と関係のない手がかりが出ていないか

In [6]:
# 分析対象の政党の発言から2万件を取り出す
s = q[q["party"] != "その他"].sample(20000, random_state=0)

# 発言 × 2〜4文字の並び の出現回数の表
vec = CountVectorizer(analyzer="char", ngram_range=(2, 4), min_df=20)
X = vec.fit_transform(s["text"])
terms = np.array(vec.get_feature_names_out())

# 党ごとの特徴語(Fightin' Words の z 値の上位15個)
for party, words in top_terms(X, terms, s["party"]).items():
    print(party, ":", "/".join(words))

みらい : デー/データ/ータ/動運/ai/自動運/動運転/自動運転/てお/ござい/ございま/ござ/ざいま/ざい/あるとか
れいわ : 障害/です/障害者/費税/消費税/改憲/すよ/んです/です。/ 資料/ですよ/。 資料/政党/ 資/すか。
中道 : こと。/ること。/イラン/イラ/備費/予備費/こと。 /自転車/自転/これは/病歴/転車/廃棄/れは/暫定予算
保守 : 再エネ/再エ/エネ/結婚/婚し/難民/さて/。 さて/ さて/エネ賦課/ネ賦課/ネ賦/再エネ賦/エネ賦/ネ賦課金
公明 : まして/りま/伺い/、また/おりま/まして、/ておりま/ります/ており/たいと/ます。/きまして/おります/おり/ります。
共産 : です/ですよ/すよ/ね。/すか。/ですか。/んですよ/すよ。/よ。/わけです/けです/ね。 /んです/ですよ。/すよね
参政 : 外国/外国人/国人/日本/受入れ/受入/国民/いった/考え/、外国/、外国人/の受入れ/の受入/えます。/国人の
国民 : いうとこ/うところ/うとこ/というと/いうと/とうござ/とうご/とう/りがとう/がとう/ありがと/がとうご/りがと/ありが/がと
社民 : 装備/銀行/条約/産業技術/産業技/業技術/業技/の装備/原発/の装/スルガ銀/ルガ銀/ルガ銀行/武器/ガ銀行
立憲 : です/こと。/んです/ること。/こと。 /んで/すか/ですか/か。/すか。/すよ/ですか。/と。 /ですよ/年金
維新 : なと/ですけ/かなと/改革/すけ/んで/いう/んです/ですけれ/首都/んですけ/なという/副首都/副首/なとい
自民 : でありま/であり/ます。/ります/りま/であ/あります/ります。/ます/あり/ありま/ござ/ございま/ござい/ざいま


**結果**: 上位の多くは**話し方**だった。自民「であります」、公明「まして」「おります」、共産「ですよ」「んですよ」、立憲「ですか」、国民「ありがとう」「というところ」、維新「かなと」「ですけれ」。話題は、みらい「データ」「自動運転」、保守「再エネ」「難民」、参政「外国人」「受入れ」、維新「改革」「副首都」など一部。党名そのものは出ていない(れいわの「政党」は、隠した結果の「政党」)。

次に、会派の名前が本文に残っていないかを数える。

In [7]:
names = set()
for g in q["speakerGroup"].unique():
    names.add(g)
    names.update(p for p in g.split("・") if len(p) >= 2)   # 「新緑風会」「無所属クラブ」など

left = {n: q["text"].str.contains(n, regex=False).sum() for n in names}
pd.Series(left)[lambda s: s > 0].sort_values(ascending=False)

国民の声         211
無所属          133
各派に属しない議員     38
無所属の会          3
dtype: int64

**結果**: 本文に残っている会派の名前は「国民の声」211・「無所属」133・「各派に属しない議員」38・「無所属の会」3 だけ。「国民の声」「無所属」は一般の言い回し(「国民の声を聞く」など)としても使われるので、多くは会派の名前ではない。

**見つけたこと**: 最初の作業では「有志の会」159・「沖縄の風」110 なども残っていた → 会派の名前を隠す一覧に足した。

発言の最初の自己紹介(「〇〇の△△です」)で、何が残っているかを見る。

In [8]:
head = q["text"].str[:80]
intro = head.str.extract(r"([^\s、。]+?)の[^\s、。]{2,8}(?:です|でございます)。")[0]
intro.value_counts().head(30)

0
政党                 312
政党会派                61
そ                   33
最後                  27
次                   24
愛媛県選出               19
政党代表                16
衆議院議員               15
無所属                 15
政党政調会長              10
参議院幹事長              10
続いて                  9
政党・政党・無所属会派          9
こ                    9
石川県能登半島              9
政党・政党・無所属            9
私                    8
広島県                  8
二点目                  8
東京十六区選出              5
初めて                  5
茨城県選出                5
略称民主党                5
神奈川県選出               5
参議院議員                5
長野五区                 5
お手元に配付されております案文      5
長野三区                 4
大臣                   4
今                    4
Name: count, dtype: int64

**結果**: 「政党」312件(自己紹介の党名は隠せている)。ほかは「愛媛県選出」「長野五区」のような**選挙区の自己紹介**や、「最後」「次」のような普通の言葉。選挙区は地域の手がかりになるが、党の手がかりではないので残した。

**見つけたこと**: 最初の作業では「政党」が 7,348件あった → 自己紹介の文(「政党の△△です」)ごと消すようにしたので、312件に減った(残りは、自己紹介の形が少し違うもの)。

次は、漢字・カタカナを含む並びだけに絞って、主な9党の上位100個を見る(ひらがなだけの話し方の並びを外して、話題を見るため)。

In [9]:
content = np.array([bool(re.search(r"[一-龥ァ-ヶ]", t)) for t in terms])

for party in ["立憲", "国民", "自民", "維新", "共産", "公明", "中道", "参政", "れいわ"]:
    z = fightin_words(X, (s["party"] == party).to_numpy())
    z[~content] = -np.inf                      # ひらがなだけの並びは除く
    top = terms[np.argsort(z)[::-1][:100]]
    print(party, ":", "/".join(top), "\n")

立憲 : 年金/修正/大臣、/臣、/大臣/職員/、大臣、/皆さ/皆さん/、大臣/の皆さ/と。 十/の皆さん/言っ/。 十/別姓/ 十/年金の/基金/努める/資料/厚生年/生年金/厚生年金/分か/百億/生年/の修正/の修/総理/職員の/委員/夫婦別姓/婦別姓/言って/関税/修正案/億円/夫婦/決議/努めるこ/会計/婦別/夫婦別/漁業/的夫婦/的夫/任用/的夫婦別/帯決/附帯決/附帯決議/帯決議/る附/年金が/附帯/選択的夫/択的夫/択的夫婦/な措置を/私は/分からな/る附帯決/を講ず/礎年/基礎年金/基礎年/礎年金/る附帯/底上げ/底上/僕は/置を講ず/分から/保育/を講ずる/言ってい/教職/講ずるこ/育士/保育士/議案/講ず/一 /この間/講ずる/理事/二 /努め/三 /書いてあ/資料の/教職員/選択的/択的/ホス/修正を/用職/用職員/被害 

国民 : 相続/ガソリ/ソリン/ガソリン/資産/住民投/住民投票/ソリ/ガソ/ンテリジ/ンテリ/インテリ/テリジ/テリジェ/別市/薬価/ェンス/リジェン/手取りを/ジェンス/リジェ/リジ/テリ/インテ/取りを増/りを増/りを増や/教え/電気/山形/暫定税率/定税率/定税/暫定税/お教/お教え/手取り/広告/中で/七十八万/手取/ソリンの/都市/、要は/、インテ/所得制限/得制/得制限/所得制/教えて/民投票/七十八/是非/リンの/河期/氷河/氷河期/お教えい/教えい/教えいた/百七十八/ンテ/税率/料金/定税率の/形県/山形県/民投/要は/ェン/火力/電力の/取りを/、ガ/伺いでき/伺いで/お伺いで/その点/と思って/職氷河期/就職氷河/就職氷/職氷/職氷河/税率の廃/率の廃/率の廃止/質問/と思っ/期世代/期世/氷河期世/河期世代/河期世/療的ケア/療的ケ/医療的ケ/の壁/的ケア/的ケ 

自民 : 地域/我が国/我が/が国/重要/我が国の/が国の/国旗/歯科/地方/、我が国/成長/の地/、我が/、地/教師/地元/国の/経済/取組/感謝/、我/先生/重要で/いと存/いと存じ/国旗を/安全保障/安全保/全保障/全保/たいと存/お伺い/お伺/要であ/幹線/旗を/と存じ/申し上/申し上げ/重要であ/と存じま/し上/強靱/と存/し上げ/靱化/強靱化/整備/山間地/間地/人口/中山間地/新幹/新幹線/様方/皆様方/要で/戦略/安全

**結果**: 政策の話題が中心になった。
- 立憲「年金」「修正案」「夫婦別姓」「附帯決議」「関税」
- 国民「ガソリン」「暫定税率」「インテリジェンス」「手取り」「氷河期」
- 自民「地域」「我が国」「国旗」「地方創生」「強靱化」
- 維新「改革」「副首都」「無償化」「連立」「社会保険料」

人の名前は出ていない。ただし、自民の「申し上げ」「存じ」のような改まった話し方は残る。

党によって件数が大きく違う(立憲 2.4万・社民 388)ので、党ごとに最大 3,000件にそろえて、全党でやり直す。

In [10]:
# 党ごとに最大3000件ずつ(小さな党は全件)
s = (q[q["party"] != "その他"]
     .sample(frac=1, random_state=0)      # 全体をランダムに並べ替える
     .groupby("party")
     .head(3000))                         # 党ごとに先頭3000件
print(s["party"].value_counts())

vec = CountVectorizer(analyzer="char", ngram_range=(2, 4), min_df=20)
X = vec.fit_transform(s["text"])
terms = np.array(vec.get_feature_names_out())
content = np.array([bool(re.search(r"[一-龥ァ-ヶ]", t)) for t in terms])

for party in s["party"].value_counts().index:      # 全党
    z = fightin_words(X, (s["party"] == party).to_numpy())
    z[~content] = -np.inf
    top = terms[np.argsort(z)[::-1][:100]]
    print(party, ":", "/".join(top), "\n")

party
国民     3000
中道     3000
立憲     3000
自民     3000
共産     3000
公明     3000
れいわ    3000
維新     3000
参政     3000
みらい    1451
保守      589
社民      388
Name: count, dtype: int64
国民 : インテリ/ンテリジ/テリジェ/テリジ/ンテリ/ェンス/テリ/インテ/リジェン/ジェンス/リジェ/リジ/ガソリ/ガソリン/ソリン/ソリ/定税率/定税/暫定税/暫定税率/薬価/ガソ/特別市/相続/別市/ェン/住民投票/住民投/取りを増/りを増や/りを増/手取りを/ソリンの/教え/山形/ジェン/リンの/、インテ/お教え/七十八万/お教/氷河期/氷河/河期/税率/定税率の/中で/手取り/所得制/所得制限/得制/得制限/職氷河期/就職氷/就職氷河/職氷河/職氷/御説明/御説/教えて/手取/の暫定税/浄化/是非/鉄道/百七十八/リンの暫/電気/七十八/教えい/お教えい/教えいた/ンの暫定/ンの暫/広告/暫定/資産/、要は/山形県/形県/ンテ/河期世/期世/期世代/河期世代/氷河期世/思って/率の廃/税率の廃/率の廃止/その点/の暫/の暫定/取りを/の壁/、ガソ/、ガソリ/続税/相続税/思っ 

中道 : イラン/イラ/病歴/ナフサ/フサ/ナフ/答弁/教会/弁を/一教/答弁を/統一教/一教会/統一教会/トス/ージ目/ジ目/ページ目/住所/ピーチ/スピーチ/統一/コード/附帯/ヘイ/附帯決/帯決議/帯決/附帯決議/名前/ヘイト/、名前/、イラ/、イラン/私は/ルムズ/ホルム/ホルムズ/イトス/イトスピ/トスピ/ヘイトス/トスピー/備費/ムズ/予備費/御答/御答弁/シンナ/シンナー/ルム/御答弁を/プラスチ/暫定予/暫定予算/受渡/定予算/チック/定予/ラスチッ/ラスチ/スチッ/スチック/受渡し/ラン情勢/イラン情/ラン情/、名/名前と/ホル/情勢/ン情勢/ン情/意味にお/味にお/、御答/、御答弁/うに思っ/意味に/弁をお/答弁をお/原油/予備/金利/審議/味に/弁をお願/に思っ/陳列/サナ/サナエ/ナエ/決議/恐ら/恐らく/ムズ海/ムズ海峡/ズ海峡/ルムズ海/ズ海 

立憲 : 年金/大臣/大臣、/修正/臣、/関税/、大臣、/

**結果**(件数をそろえた後): 国民「インテリジェンス」「ガソリン」「暫定税率」、中道「イラン」「ナフサ」「統一教会」「ホルムズ」、立憲「年金」「関税」「会計年度任用職員」、維新「改革」「副首都」「無償化」「連立」、共産「軍事」「米軍」、参政「外国人」「受入れ」、保守「外国人」「難民」「再エネ」など、**その党が国会で力を入れた話題**が並ぶ。

## 3. 委員長報告

委員長報告(「〜委員会における審査の経過及び結果を御報告申し上げます」)は、委員会の結果の報告で、議員の考えではない。残っていないかを確かめる。まず、ひらがなの「ご報告」で探す。

In [11]:
rep = q[q["speech"].str.contains("ご報告申し上げます", regex=False)]
print(len(rep))
rep["speech"].str[:30].value_counts().head(15)

0


Series([], Name: count, dtype: int64)

**結果**: 0件。次に漢字の「御報告申し上げます」で探す。

In [12]:
rep = q[q["speech"].str.contains("御報告申し上げます", regex=False)]
print(len(rep))
rep["speech"].str[:100].value_counts().head(15)

11


speech
○竹内譲君　ただいま議題となりました両案件について申し上げます。\r\n　まず、地域人口の急減に対処するための特定地域づくり事業の推進に関する法律の一部を改正する法律案につきまして、提案の趣旨及び内容を御      1
○御法川信英君　ただいま議題となりました三法律案につきまして申し上げます。\r\n　まず、土地改良法等の一部を改正する法律案につきまして、農林水産委員会における審査の経過及び結果を御報告申し上げます。\r\n    1
○有村治子君　情報監視審査会は、去る六月六日、審査会規程第二十二条第一項に基づき、本審査会における調査及び審査の経過及び結果を記載した年次報告書を作成し、会長から議長に提出をいたしました。\r\n　本報告      1
○伊藤達也君　情報監視審査会は、去る三日、審査会規程に基づき、令和六年年次報告書を取りまとめ、額賀議長に提出をいたしました。\r\n　以下、その概要を御報告をいたします。\r\n　今回の報告書の対象期間は、令    1
○石井章君　ただいま議題となりました法律案につきまして、消費者問題に関する特別委員会における審査の経過と結果の御報告を申し上げます。\r\n　本法律案は、公益通報者の範囲を拡大するとともに、公益通報をした      1
○藤井比早之君　ただいま議題となりました両法律案につきまして申し上げます。\r\n　まず、家畜伝染病予防法の一部を改正する法律案につきまして、農林水産委員会における審査の経過及び結果を御報告申し上げます。      1
○船田元君　情報監視審査会は、去る二十五日、審査会規程に基づき、令和七年年次報告書を取りまとめ、森議長に提出いたしました。\r\n　以下、その概要を御報告いたします。\r\n　今回の報告書の対象期間は、令和七    1
○山下貴司君　ただいま議題となりました両法律案について申し上げます。\r\n　まず、科学技術・イノベーション創出の活性化に関する法律の一部を改正する法律案について、内閣委員会における審査の経過及び結果を御      1
○櫻井充君　ただいま議題となりました法律案につきまして、政治改革に関する特別委員会における審査の経過と結果の御報告を申し上げます。\r\n　本法律案は、選挙に関するインターネット等の利用の状況に対応するた      1
○芳賀

**結果**: 11件。読むと、情報監視審査会の年次報告、行政監視委員会の報告、「まず、〜」と2つの法律案をまとめて報告する委員長報告(「御報告申し上げます」が 200字より後ろに来る)だった。

**見つけたこと**: 最初の作業では 328件あり、自民の特徴語の上位が「御報告申し上げます」「法律案」になっていた → 「発言の最初の200字以内に『御報告申し上げ / いたし』がある発言」を除く決まり(`REPORT`)を前処理に入れた。

次のセルで、その決まりに当たる発言が 0件になったことを確かめる。

In [13]:
REPORT = r"^[^\n]{0,200}?御報告(?:申し上げ|いたし)"
is_rep = q["speech"].str.contains(REPORT)
print("該当:", is_rep.sum())
print(q[is_rep]["party"].value_counts())
q[is_rep]["speech"].str[:60].head(10)      # 0件なら空のまま(前処理で委員長報告を除いた後は 0件になる)

該当: 0
Series([], Name: count, dtype: int64)


Series([], Name: speech, dtype: str)

**結果**: 0件 → 決まりは効いている。残った 11件(上のセル)は、報告が 200字より後ろにあるもので、全体(88,297件)から見るとわずか。

## 4. 一般の言葉と重なる党の略称

「中道」を含む発言を探す。中道改革連合の略称だが、一般の言葉でもある。

In [14]:
has = q["text"].str.contains("中道", regex=False)
print("中道を含む発言:", has.sum())
print(q[has]["party"].value_counts())
q[has & (q["party"] == "自民")]["text"].str.extract(r"(.{0,15}中道.{0,15})")[0]

中道を含む発言: 48
party
公明     9
国民     8
自民     7
維新     7
立憲     4
共産     4
みらい    3
その他    2
れいわ    2
参政     1
社民     1
Name: count, dtype: int64


472       し、国民世論は左右両極に分化、中道勢力が縮小していくことで、政治
13138     ございます。そういった意味で、中道的な日本のエネルギー政策という
13190     ただきました。その上で、日本は中道的な対応を続けていくことが大切
141180    た。政党と政党さんが合併をして中道になった。そして、政党、政党さ
156035    私が調べたところだと、いわゆる中道左派と呼ばれるような方でござい
174752    、立場の隔たりという意味では、中道、国民の皆様からは規制強化の法
181129    考人から、欧州統合を支えてきた中道右派、中道左派の二大勢力に代わ
Name: 0, dtype: str

**結果**: 48件。自民の例を見ると「中道的な日本のエネルギー政策」「中道左派」「中道右派」のような**一般の言葉**が多く、これは残すのが正しい。「合併をして中道になった」の「中道」は党の略称だが、後ろに「提出」「代表」などが続かないので、今の決まりでは隠れていない(隠し漏れの例)。

**見つけたこと**: 最初の作業では 187件あった。一律に「中道」を隠すと一般の言葉まで消えるので、次のセルの関数のように、**党を指す文脈のときだけ**隠すことにした(後ろに「提出」「さん」「案」「の〇〇代表」などが続くとき、または「政党、中道、国民」のような並びのとき)。中道・国民・参政・みらいの4つが対象。この関数は `src/preprocess/filters.py` に入っている。

In [15]:
AMBIG = "中道|国民|参政|みらい"   # 一般の言葉と重なる党の略称

# 後ろの文脈で判断:「中道提出」「国民さん」「中道の小川代表」など
CTX_AFTER = re.compile(rf"({AMBIG})(?=提出|さん|案|の[一-龥]{{1,4}}(?:代表|幹事長|幹事|議員|委員|提出者))")
# 並びで判断:「政党、中道」「中道、政党」
LIST_BEFORE = re.compile(rf"(?<=政党、)({AMBIG})")
LIST_AFTER = re.compile(rf"({AMBIG})(?=、政党)")


def mask_ambiguous(text):
    """一般の言葉と重なる党の略称を、党を指す文脈のときだけマスクする"""
    text = CTX_AFTER.sub("政党", text)
    for _ in range(5):                    # 「政党、中道、国民、参政」のような連続に対応するため、くり返す
        new = LIST_AFTER.sub("政党", LIST_BEFORE.sub("政党", text))
        if new == text:
            break
        text = new
    return text

## 5. 人の名前

「高市総理」「石破総理」のような名前は、時期(どの内閣か)や、どの大臣に質問したかの手がかりになり、立場とは関係ない。肩書き(総理・大臣・代表・さん など)の前に来る1〜3文字の漢字を数える。

In [16]:
titles = q["text"].str.findall(r"([一-龥]{1,3})(?:総理|大臣|代表|幹事長|官房長官|さん)")
pd.Series([n for lst in titles for n in lst]).value_counts().head(40)

皆      12702
副       1472
外務      1235
政党      1187
今       1158
患者      1074
財務      1024
法務       853
総務       761
防衛       701
内閣       675
生労働      592
子        565
総理       556
厚労       513
経産       406
文科       362
環境       355
是非       343
母        302
部科学      274
安倍       256
担当       255
林水産      242
子供       236
土交通      230
国務       230
医者       224
済産業      218
前        217
農家       214
木外務      195
屋外務      185
父        177
農水       174
親御       174
国交       167
事業者      149
環境省      145
職員       134
Name: count, dtype: int64

**結果**: 上位は「皆(さん)」「副(大臣)」「外務(大臣)」「財務」など、肩書きの一部や普通の言葉。人の名字は「安倍」256 が一番多く、今の議員の名字(高市・石破など)は上位に出てこない → 隠せている(「安倍」は元総理で、今の議員の名字の一覧に入っていないため残る)。

**見つけたこと**: 最初の作業では、高市 2,320・石破 1,960・赤澤 1,047・小泉 670 などが上位だった → 議員の名字の一覧を作り、名字を隠すようにした(以下のセル)。

「委員・議員・君」の前も数える。

In [17]:
titles = q["text"].str.findall(r"([一-龥]{1,3})(?:委員|議員|君)")
pd.Series([n for lst in titles for n in lst]).value_counts().head(40)

予算     1912
国会     1376
教育      524
衆議院     465
本       462
家公安     420
当       371
参議院     358
正取引     351
内閣      325
法務      294
理事      272
第三者     271
生労働     271
特別      239
規制      212
地方      210
決算      209
報保護     206
理大臣     201
各       196
院予算     186
総務      186
専門      186
経営      181
力規制     175
務大臣     159
林水産     146
外務      140
務金融     134
環境      133
同僚      132
厚労      127
挙管理     125
済産業     121
土交通     112
品安全     111
別撤廃     110
評価      107
交防衛     104
Name: count, dtype: int64

**結果**: 委員会の名前(予算・国会・教育など)や肩書きの一部が多く、人の名字は上位に無い。

以下の5つのセルは、名字の一覧の作り方の確認。発言者の氏名と、衆議院の会議録の見出し(「○長妻委員」「○赤澤国務大臣」)から名字を取り出す。

In [18]:
# 氏名(発言者の列から)
full_names = sorted(df["speaker"].dropna().unique(), key=len, reverse=True)
full_names = [n for n in full_names if n != "会議録情報" and len(n) >= 3]
print("氏名の数:", len(full_names))

# 名字(衆議院の見出し「○長妻委員」「○赤澤国務大臣」から)
HEAD = re.compile(
    r"^○([一-龥々ヶぁ-んァ-ヶー]{1,6})(?:（[^）]*）)?"
    r"(?:委員|君|国務大臣|内閣総理大臣|副大臣|大臣政務官|政府特別補佐人)"
)
sur = df["speech"].str.extract(HEAD)[0]
pairs = pd.DataFrame({"speaker": df["speaker"], "surname": sur}).dropna().drop_duplicates()

# 氏名の先頭と一致する名字だけを残す(見出しの読み取り誤りを除く)
pairs = pairs[[n.startswith(s) for n, s in zip(pairs["speaker"], pairs["surname"])]]
surnames = sorted(pairs["surname"].unique(), key=len, reverse=True)
print("名字の数:", len(surnames))
print(pairs.head(20))

氏名の数: 2177
名字の数: 983
    speaker surname
1       安住淳      安住
4     小野寺五典     小野寺
5       石破茂      石破
7       城内実      城内
9      武藤容治      武藤
23      江藤拓      江藤
29    中曽根康隆     中曽根
34      岩屋毅      岩屋
36      中谷元      中谷
45     木原誠二      木原
52     赤澤亮正      赤澤
63     塩崎彰久      塩崎
72     加藤勝信      加藤
74     福岡資麿      福岡
87      長妻昭      長妻
132     城井崇      城井
139    あべ俊子      あべ
168   源馬謙太郎      源馬
202      階猛       階
241   奥野総一郎      奥野


**結果**: 氏名 2,177、名字 983。

議員(会派の付いた発言者)のうち、見出しから名字が取れなかった議員を数える。

In [19]:
members = set(df.loc[df["speakerGroup"].notna(), "speaker"])
no_surname = sorted(members - set(pairs["speaker"]))
print("議員の数:", len(members))
print("名字が取れなかった議員:", len(no_surname))
print(no_surname[:40])

議員の数: 895
名字が取れなかった議員: 7
['世耕弘成', '中曽根弘文', '森英介', '玄葉光一郎', '石井啓一', '逢沢一郎', '額賀福志郎']


**結果**: 議員 895人のうち 7人(世耕弘成・玄葉光一郎など)。ただし、次のセルのように、見出しから氏名まるごとを名字として取ってしまった組がある。

In [20]:
full_as_sur = pairs[pairs["speaker"] == pairs["surname"]]
print("氏名まるごとが名字になっている:", len(full_as_sur))
full_as_sur.head(10)

氏名まるごとが名字になっている: 516


,speaker,surname
379,竹谷とし子,竹谷とし子
383,浅田均,浅田均
387,川合孝典,川合孝典
393,小池晃,小池晃
396,熊谷裕人,熊谷裕人
399,比嘉奈津美,比嘉奈津美
403,山本太郎,山本太郎
436,重徳和彦,重徳和彦
441,西岡秀子,西岡秀子
446,斉藤鉄夫,斉藤鉄夫


**結果**: 氏名まるごとが名字になっている組が 516(参議院の見出しは「○小池晃君」のように氏名で書かれるため)。これを除いて数え直し、ほかの議員の名字(「上田」「伊藤」など)で補えるかを試す。

In [21]:
no_surname = sorted(members - set(pairs.loc[pairs["speaker"] != pairs["surname"], "speaker"]))
print("名字が取れなかった議員:", len(no_surname))

known = sorted(set(pairs.loc[pairs["speaker"] != pairs["surname"], "surname"]), key=len, reverse=True)

extra = []
for name in no_surname:
    for sur in known:                      # s は党ごとの発言の表の名前なので使わない
        if name.startswith(sur) and len(sur) < len(name):
            extra.append((name, sur))
            break

extra = pd.DataFrame(extra, columns=["speaker", "surname"])
print("補えた議員:", len(extra), "/", len(no_surname))
print(extra.head(20))
print("それでも残った議員:", sorted(set(no_surname) - set(extra["speaker"])))

名字が取れなかった議員: 273
補えた議員: 137 / 273
   speaker surname
0      上田勇      上田
1     上田清司      上田
2    上野ほたる      上野
3     上野通子      上野
4     下野六太      下野
5    中曽根弘文     中曽根
6     中田優子      中田
7     中西祐介      中西
8     井上哲士      井上
9     井上義行      井上
10    伊藤孝恵      伊藤
11    伊藤孝江      伊藤
12     伊藤岳      伊藤
13    伊藤辰夫      伊藤
14    伊藤達也      伊藤
15  佐々木さやか     佐々木
16   佐々木りえ     佐々木
17   佐々木雅文     佐々木
18    佐藤信秋      佐藤
19     佐藤啓      佐藤
それでも残った議員: ['いんどう周作', 'かごしま彰宏', 'かまやち敏', 'こやり隆史', 'ながえ孝子', 'ラサール石井', '三上えり', '三宅伸吾', '三浦信祐', '三浦靖', '上月良祐', '世耕弘成', '中条きよし', '串田誠一', '仁比聡平', '伊勢崎賢治', '伊波洋一', '倉林明子', '出川桃子', '初鹿野裕樹', '加田裕之', '勝部賢志', '北村晴男', '北村経夫', '古屋圭司', '古庄玄知', '司隆史', '嘉田由紀子', '堂故茂', '堂込麻紀子', '塩入清香', '塩村あやか', '塩田博昭', '大家敏志', '大椿ゆうこ', '大津力', '大門実紀史', '天畠大輔', '奥村政佳', '奥村祥大', '奥田ふみよ', '安江伸夫', '安達悠司', '安野貴博', '宮出千慧', '宮口治子', '宮本周司', '宮本和宏', '宮沢洋一', '小島とも子', '小沢雅仁', '小沼巧', '小西洋之', '岡崎太', '岩本剛人', '岩本麻奈', '岩渕友', '広田一', '庭田幸恵', '徳永エリ', '打越さく良', '新妻秀規', '新実彰平', '星北斗', '有村治子', '望月良男', '朝日健太郎', '木戸口英司'

**結果**: 除いて数え直すと、名字が取れない議員は 273人。ほかの議員の名字で補えたのは 137人。残り(ひらがなを含む名前や、珍しい名字の議員)は、名字と名前の境目が分からないので、`build_member_table` で推測する(次のセル)。

In [22]:
from io import StringIO

members_tbl = build_member_table(df)          # 除外する前の df を使う
print(members_tbl["source"].value_counts())

# 前処理が読む data/external/member_names.csv と同じかを確かめる(ここでは上書きしない)
now = pd.read_csv(StringIO(members_tbl.to_csv(index=False)))
saved = pd.read_csv("../data/external/member_names.csv")
print("今作った表:", len(now), "人 / 保存してある表:", len(saved), "人 → 同じか:", now.equals(saved))
# 作り直して保存するときだけ、次の行の # を外す
# members_tbl.to_csv("../data/external/member_names.csv", index=False)

source
guess     855
known     698
header    628
none        3
Name: count, dtype: int64
今作った表: 2184 人 / 保存してある表: 2184 人 → 同じか: True


**結果**: 名字の出どころは、推測 855・ほかの議員の名字 698・見出し 628・不明 3。今作った表(2,184人)は、前処理が読む `data/external/member_names.csv` と**同じ**だった(このセルでは上書きしない)。

次のセルは、名字の決まり・人の名前を隠す関数・委員長報告の決まりが `src/preprocess/filters.py` に入っていることの確認。

In [23]:
import src.preprocess.filters as f
print("SURNAME_RE" in dir(f), "mask_person_names" in dir(f), "REPORT" in dir(f))
print("議員名の一覧:", "読み込み済み" if f.FULL_NAME_RE is not None else "なし")

True True True
議員名の一覧: 読み込み済み


**結果**: 3つとも入っていて、議員名の一覧も読み込めている。

隠す**前**の本文(`text_nomask`)で、名字がどれくらい出てくるかを数える(隠す対象がちゃんと見つかるかの確認)。

In [24]:
hits = q["text_nomask"].str.findall(SURNAME_RE)
pd.Series([h for lst in hits for h in lst]).value_counts().head(40)

高市     2327
石破     2031
赤澤     1051
小泉      726
片山      516
加藤      481
茂木      475
松本      448
鈴木      426
林       372
上野      368
福岡      341
平       331
黄川田     328
城内      313
小野田     301
三原      291
岩屋      289
金子      280
武藤      271
伊東      270
江藤      269
坂井      265
岸田      253
中野      223
村上      218
中谷      168
あべ      164
木原      161
野田      146
古賀      127
あかま     123
浅尾      120
玉木      114
西田      110
高橋      108
牧野      107
石原      105
吉村      102
前川       96
Name: count, dtype: int64

**結果**: 高市 2,327・石破 2,031・赤澤 1,051・小泉 726 など。隠す前の本文なので、出てくるのが正しい。これらが、隠した後の本文(5. の最初のセル)では上位から消えている。

## 6. 気になった言葉を、文脈ごと読んで確かめる

特徴語の上位で気になった言葉を、2. の件数をそろえた発言(`s`)の中で探し、前後の文と一緒に読む。後半は、隠した結果の「政党」がどんな形で残っているかを数える。

In [25]:
def show(party, pat, n=3):
    hit = s[(s["party"] == party) & s["text"].str.contains(pat)]
    print(f"[{party}] {pat}: {len(hit)}件")
    for t in hit["text"].head(n):
        i = t.find(pat)
        print("   …", t[max(0, i - 60): i + 30])

show("自民", "質疑は終了")
show("自民", "緊急集会")
show("立憲", "電磁的記録")
show("中道", "連合")          # ①の修正後は、ほぼ0件になるはず
show("れいわ", "政党")

for pat in ["政党、議員", "政党の議員", "我々政党", "政党は"]:
    print(pat, s[s["text"].str.contains(pat)]["party"].value_counts().head(5).to_dict())

[自民] 質疑は終了: 0件
[自民] 緊急集会: 15件
   …  参議院の緊急集会は、現行憲法上唯一の緊急事態条項であり、参議院の重要
   … は、同時に閉会となる。」と、これ本文でこう書いています。ただし書で、「但し、内閣は、国に緊急の必要があるときは、参議院の緊急集会を求めることができる。」と、こういうふうに書いていま
   …  本日は、参議院の緊急集会と議員任期特例のすみ分けについて申し上げます。 まず
[立憲] 電磁的記録: 19件
   … ャットで何かの拍子にあの先生は嫌いだと書いたら、永遠にそれが残っているということが起こり得るわけですよ。しかも、それが、電磁的記録提供命令が不当だと取り消されたって削除されない。 
   … 局長が出された例は覚醒剤なんですよ。法禁物なんですよ。情報だけ特化するわけにいかないとおっしゃいましたけれども、覚醒剤と電磁的記録の中に入っている情報は価値が違うのは当たり前じゃな
   … これまでの答弁からしますと、要するに、犯罪と関連性のない個人情報が電磁的記録提供命令あるいは差押えにおいて一切収集されないとい
[中道] 連合: 20件
   … 取組、外務大臣を始め外務省の皆様の御尽力に敬意を表したいと思います。御苦労さまでございます。 質問を変えます。 万国郵便連合憲章の件でございますが、この万国郵便連合、UPU、この役
   … これは、警察と国家情報局の関係だったり、あるいは公安調査庁との関係なので、連合審査あたりで本来審議する必要があると思いますので、詳しい
   … もかかってくる。そうすると、オウム真理教でも十五年、二十年、清算までにかかっているんだけれども、恐らくこの統一教会、家庭連合の清算についても十年単位で時間がかかるというふうに思うと
[れいわ] 政党: 504件
   … ありがとうございます。 大演説していただいて恐縮なんですけれども、私たち政党、別に、国庫負担を減らすのは関係なくて、むしろ増えても全
   … めなきゃいけないんですよ。六月十二日の幹事会で、改憲五会派で共同で、憲法改正の骨子案、緊急事態条項の骨子案を出すんやと。政党、政党、政党、政党、政党ですよね。こんなことはやっちゃい
   … ただきました。これを残したままですと、障害者権利条約の差別にも当たる、違反にも当たるんじゃ

**結果**
- 自民「質疑は終了」: 0件(司会の決まり文句は消えている)
- 自民「緊急集会」: 15件。参議院の緊急集会(憲法の緊急事態の話)についての議論で、政策の話題
- 立憲「電磁的記録」: 19件。法案(電磁的記録の提供命令)の話題
- 中道「連合」: 20件。「万国郵便連合」「連合審査」「家庭連合(統一教会)」など一般の言葉。党名「中道改革連合」は隠れている
- れいわ「政党」: 504件。「政党、政党、政党…」のように、党名を並べて批判した部分が「政党」になったもの
- 「政党は」「政党の議員」は、参政・れいわに多い(自分の党や他の党を主語にして話すことが多い)

**見つけたこと**: 最初の作業では、自民の特徴語に「質疑は終了」が出ていた。次のセルで、誰が言っていたかを確かめる。

In [26]:
m = q["speech"].str.contains("質疑は終了いたしました")
head = q.loc[m, "speech"].str.extract(r"^(○\S+?)[\s　]")[0]
print(m.sum(), "件")
print(head.str.replace(r"（[^）]*）", "", regex=True).str[-4:].value_counts().head(15))

0 件
Series([], Name: count, dtype: int64)


**結果**: 0件。

**見つけたこと**: 最初の作業では 183件あり、言っていたのは分科会の**主査**・公聴会の**座長**・理事など**司会役**だった(委員長以外の司会役)→ 司会役の発言を除く決まりを入れた。

直した後に、司会役の発言・自己紹介が消えたか、消しすぎていないかを確かめる。

In [27]:
# ① 司会役(主査・理事・座長)が消えたか
m = q["speech"].str.contains("質疑は終了いたしました")
print("質疑は終了:", m.sum(), "件")

# ② 自己紹介と「我々政党」が消えたか
for pat in ["政党の議員です", "政党、議員です", "政党の議員でございます", "我々政党"]:
    print(pat, q["text"].str.contains(pat).sum(), "件")

# ③ 自己紹介を消しすぎていないか(例を3件)
hit = q[q["text_nomask"].str.contains("の議員の皆さん")]
for t in hit["text"].head(3):
    i = t.find("議員の皆さん")
    print("   …", t[max(0, i - 40): i + 20])

質疑は終了: 0 件
政党の議員です 0 件
政党、議員です 0 件
政党の議員でございます 0 件
我々政党 0 件
   … は、積極的に不記載の指示はしていないと言っている。一方で、政倫審に出られた多くの議員の皆さんは、指示を受けてやった、秘書
   … すけれども。 大臣、十年後も恐らくこの審議をきっとするんだと思います。そのときの議員の皆さんが、この十年間の新たな受給権
   … と出してこの成果を国民の皆さんに示すことが大事なんだというようなこともたくさんの議員の皆さんから言われました。 その中で


**結果**: 「質疑は終了」0件、自己紹介の決まり文句(「政党の議員です」など)0件、「我々政党」0件。「議員の皆さん」のような普通の言葉は残っている → 消しすぎていない。

本文を作る関数の例を試す。

In [28]:
print("主査" in f.CHAIR)
print(prepare_text("○田中委員　自由民主党の田中太郎です。我々日本維新の会は反対です。", "田中太郎", "自民"))

True
我々は反対です。


**結果**: 司会役の一覧(`CHAIR`)に「主査」が入っている。「○田中委員 自由民主党の田中太郎です。我々日本維新の会は反対です。」→「我々は反対です。」。見出し・自己紹介が消え、党名も消えた。

## 7. 会派 → 党の対応表

前処理(`scripts/02_preprocess.py`)が読む対応表を表示する(ここでは書き換えない)。

In [29]:
# 会派 → 党の対応表(data/external/group_party_map.csv。前処理はこの表を読む)
print(pd.read_csv("../data/external/group_party_map.csv").to_string(index=False))

speakerGroup party
 自由民主党・無所属の会    自民
       自由民主党    自民
   立憲民主党・無所属    立憲
 立憲民主・社民・無所属    立憲
    立憲民主・無所属    立憲
  中道改革連合・無所属    中道
         公明党    公明
      日本維新の会    維新
国民民主党・無所属クラブ    国民
  国民民主党・新緑風会    国民
       日本共産党    共産
      れいわ新選組   れいわ
         参政党    参政
       日本保守党    保守
       社会民主党    社民
      チームみらい   みらい
チームみらい・無所属の会   みらい
  自由民主党・国民の声    自民


**結果**: 18の会派の名前を、12の党に対応させている(例: 「自由民主党・無所属の会」「自由民主党」「自由民主党・国民の声」→ 自民)。対応表に無い会派は「その他」。

## まとめ

- 前処理した本文には、党名・会派名・自己紹介・今の議員の名字・委員長報告・司会役の決まり文句が、ほぼ残っていない
- 隠した後の特徴語は、その党が国会で力を入れた**政策の話題**が中心になった
- ただし、**文末の話し方の違い**(「であります」「ですよ」)は残る。これは隠すものではないが、05 の BERT の埋め込みでは、これが主な違いになっていた
- 決めた処理: `src/preprocess/`(filters.py・members.py)、`scripts/02_preprocess.py`、対応表 `data/external/group_party_map.csv`・`member_names.csv`

**限界**
- 一般の言葉と重なる略称は、文脈の決まりで判断しているので、隠し漏れ(「合併をして中道になった」)が少し残る
- 元総理など、今の議員でない人の名字(「安倍」)は隠していない
- 委員長報告は「最初の200字以内」で探しているので、2つの法律案をまとめた報告などが少し残る(11件)